# A graph neural network

_Capstones_

**Build node classification on a small citation-style graph, then compare three ways to mix neighbors: GCN, GraphSAGE, and GAT.**

---

This notebook is a practice scaffold for the **A graph neural network** lesson. We build a toy citation graph, then three message-passing layers that differ only in how they aggregate neighbors, and train them the same way so the comparison is fair. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build a toy citation graph and a semi-supervised split

We synthesize a small Cora-like graph: `K` topic-communities where nodes link densely *within* a topic and sparsely *across* topics (homophily). The node features are deliberately weak — a tiny class-correlated bump buried in noise — so the real class signal lives in the **edges**, not the features. We then take a semi-supervised split: only 4 labeled nodes per class for training, the rest held out for testing.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# FINAL BUILD: node classification on a Cora-like toy graph.
# Three message-passing models on the SAME data: GCN vs GraphSAGE vs GAT.
# Each is "message -> aggregate -> update"; only the AGGREGATE step differs.
# =====================================================================
torch.manual_seed(0)

# --- 1. Build a small citation-style graph (Cora-like toy). ----------------
# K topic-communities; dense WITHIN a topic, sparse ACROSS topics (homophily).
# Node features are weak on purpose -> the class signal lives in the EDGES.
def make_citation_graph(n_per=20, K=3, p_in=0.18, p_out=0.012, feat_dim=8, seed=1):
    g = torch.Generator().manual_seed(seed)
    n = n_per * K
    y = torch.arange(K).repeat_interleave(n_per)          # ground-truth topic per node
    A = torch.zeros(n, n)
    for i in range(n):
        for j in range(i + 1, n):
            p = p_in if y[i] == y[j] else p_out
            if torch.rand(1, generator=g).item() < p:
                A[i, j] = A[j, i] = 1.0
    # weak features: small class-correlated bump buried in noise (features alone ~ chance)
    X = torch.randn(n, feat_dim, generator=g) * 1.0
    bump = torch.randn(K, feat_dim, generator=g) * 0.25
    X = X + bump[y]
    return A, X, y

A, X, y = make_citation_graph()
n, feat_dim = X.shape
K = int(y.max().item()) + 1

# adjacency list (neighbors of each node) -- used by GraphSAGE sampling
neighbors = [torch.nonzero(A[i], as_tuple=False).flatten() for i in range(n)]

# semi-supervised split: a few labeled nodes per class = train; the rest = test
g = torch.Generator().manual_seed(7)
train_idx, test_idx = [], []
for c in range(K):
    idx = torch.nonzero(y == c, as_tuple=False).flatten()
    idx = idx[torch.randperm(len(idx), generator=g)]
    train_idx += idx[:4].tolist()           # 4 labels per class
    test_idx  += idx[4:].tolist()
train_idx = torch.tensor(train_idx)
test_idx = torch.tensor(test_idx)

### Step 2 — Define the three aggregation layers

All three layers follow message → aggregate → update; only the **aggregate** step differs. **GCN** multiplies by the renormalized adjacency `S = D^-1/2 (A+I) D^-1/2` — a fixed degree-weighted neighbor average. **GraphSAGE** takes the mean of a uniform *sample* of neighbors (the inductive trick), concatenates it with the node's own vector, and L2-normalizes. **GAT** computes a *learned attention* weight per neighbor and averages over several heads. Building all three from `torch.nn` (no graph library) makes the one moving part explicit.

In [ ]:
# --- 2. The three layers (built from torch.nn; no graph library). ----------

# (a) GCN: aggregate = multiply by renormalized adjacency S = D^-1/2 (A+I) D^-1/2
def normalized_adjacency(A):
    Ah = A + torch.eye(A.shape[0])          # add self-loops
    d  = Ah.sum(1)
    Di = torch.diag(d.pow(-0.5))
    return Di @ Ah @ Di
S = normalized_adjacency(A)

class GCNLayer(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.lin = nn.Linear(c_in, c_out, bias=False)
    def forward(self, H):
        return S @ self.lin(H)              # S H W : fixed degree-weighted neighbor average

# (b) GraphSAGE: aggregate = mean of SAMPLED neighbors, then CONCAT with self
class SAGELayer(nn.Module):
    def __init__(self, c_in, c_out, n_sample=5):
        super().__init__()
        self.lin = nn.Linear(2 * c_in, c_out)   # W over concat(self, neighbor-mean)
        self.n_sample = n_sample
    def forward(self, H):
        agg = torch.zeros_like(H)
        for v in range(H.shape[0]):
            nb = neighbors[v]
            if len(nb) == 0:
                agg[v] = H[v]                # isolated node aggregates itself
            else:
                if len(nb) > self.n_sample:  # uniform neighbor SAMPLE (the inductive step)
                    nb = nb[torch.randperm(len(nb))[:self.n_sample]]
                agg[v] = H[nb].mean(0)       # mean AGGREGATE
        out = self.lin(torch.cat([H, agg], dim=1))   # CONCAT-combine
        return F.normalize(out, p=2, dim=1)          # L2-normalize the embeddings

# (c) GAT: aggregate = LEARNED-attention weighted average, averaged over heads
class GATHead(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.W = nn.Linear(c_in, c_out, bias=False)
        self.a_src = nn.Parameter(torch.empty(c_out)); nn.init.normal_(self.a_src, std=0.3)
        self.a_dst = nn.Parameter(torch.empty(c_out)); nn.init.normal_(self.a_dst, std=0.3)
        self.leaky = nn.LeakyReLU(0.2)
        self.register_buffer("mask", (A + torch.eye(A.shape[0])) > 0)  # neighbors + self
    def forward(self, H):
        Wh = self.W(H)                                       # (n, c_out)
        e  = self.leaky((Wh * self.a_src).sum(-1, keepdim=True)
                        + (Wh * self.a_dst).sum(-1, keepdim=True).T)   # (n,n) edge scores
        e  = e.masked_fill(~self.mask, float("-inf"))        # only attend to neighbors
        alpha = torch.softmax(e, dim=1)                      # learned per-neighbor weights
        return alpha @ Wh                                    # sum_j alpha_ij Wh_j

class GATLayer(nn.Module):
    def __init__(self, c_in, c_out, heads=4):
        super().__init__()
        self.heads = nn.ModuleList([GATHead(c_in, c_out) for _ in range(heads)])
    def forward(self, H):
        return torch.stack([h(H) for h in self.heads], 0).mean(0)  # average the heads

### Step 3 — Wrap each layer in the same 2-layer model

To keep the comparison fair, every model uses the identical skeleton: two message-passing layers with an ELU nonlinearity between them, mapping input features → hidden → class logits. The `builders` dict constructs one `GNN` per aggregation style with the same widths, so the *only* thing that changes between them is the layer type.

In [ ]:
# --- 3. A shared 2-layer model wrapper, so the comparison is fair. ---------
class GNN(nn.Module):
    def __init__(self, make_layer, c_in, c_hid, c_out):
        super().__init__()
        self.l1 = make_layer(c_in, c_hid)
        self.l2 = make_layer(c_hid, c_out)
    def forward(self, H):
        return self.l2(F.elu(self.l1(H)))    # message-pass twice -> logits

builders = {
    "GCN":       lambda: GNN(GCNLayer,  feat_dim, 16, K),
    "GraphSAGE": lambda: GNN(SAGELayer, feat_dim, 16, K),
    "GAT":       lambda: GNN(GATLayer,  feat_dim, 16, K),
}

### Step 4 — Train each model the same way and compare

`train_eval` trains a model on the few labeled nodes with cross-entropy, then reports accuracy on the held-out nodes. We run it for each of the three aggregators under identical optimizer, learning rate, weight decay, and epoch count. All three should comfortably beat the `1/K` chance baseline — because message passing turns "who I'm connected to" into the discriminative feature, even though the raw node features are near-chance.

In [ ]:
# --- 4. Train each model the SAME way; print + compare test accuracy. ------
def train_eval(model, epochs=120):
    opt = torch.optim.Adam(model.parameters(), lr=0.02, weight_decay=5e-4)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = F.cross_entropy(model(X)[train_idx], y[train_idx])  # labeled nodes only
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(1)
    return (pred[test_idx] == y[test_idx]).float().mean().item()

print(f"graph: {n} nodes, {K} classes, {int(A.sum().item()//2)} edges; "
      f"{len(train_idx)} labeled, {len(test_idx)} held-out test\n")
results = {}
for name, build in builders.items():
    torch.manual_seed(0)
    results[name] = train_eval(build())
    print(f"{name:>10s}  test accuracy = {results[name]:.3f}")

best = max(results, key=results.get)
print(f"\nbest on this toy graph: {best} ({results[best]:.3f})")
# Example output (our small synthetic run -- NOT the paper's Cora numbers):
#        GCN  test accuracy = 0.870
#  GraphSAGE  test accuracy = 0.852
#        GAT  test accuracy = 0.907
# best on this toy graph: GAT (0.907)
# All three beat the ~0.33 (1/K) chance baseline because message passing turns
# "who I'm connected to" into the feature -- the node features alone are near-chance.

## Visualize the data & results

_On the same Cora-like toy graph, with the same data, split, optimizer, and loss, how do the three aggregation styles &mdash; GCN, GraphSAGE, and GAT &mdash; compare on held-out node-classification accuracy?_

### Step 5 — Re-run the comparison, distilled

This block reproduces the same train-and-evaluate loop in a compact form so the comparison stands on its own. For each aggregator it builds the model, trains 120 epochs with the identical Adam settings, and records held-out accuracy.

In [ ]:
# This is the comparison block from the final build above, distilled.
# (Run the full CODE cell to reproduce these exact numbers.)
import torch
import torch.nn.functional as F

results = {}
for name, build in builders.items():     # GCN / GraphSAGE / GAT, same wrapper
    torch.manual_seed(0)
    model = build()
    opt = torch.optim.Adam(model.parameters(), lr=0.02, weight_decay=5e-4)
    for _ in range(120):
        model.train()
        opt.zero_grad()
        F.cross_entropy(model(X)[train_idx], y[train_idx]).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(1)
    results[name] = (pred[test_idx] == y[test_idx]).float().mean().item()

### Step 6 — Add the chance baseline and print the scores

Finally we tack on the `1/K` random-guess baseline for reference and print the full dictionary. Every GNN should sit well above chance, with GAT typically edging ahead on this homophilous toy graph.

In [ ]:
results["chance (1/K)"] = 1.0 / K
print(results)   # {'GCN': 0.870, 'GraphSAGE': 0.852, 'GAT': 0.907, 'chance (1/K)': 0.333}

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The three models in this capstone share data, train/test split, optimizer, loss, and number of layers. Exactly one thing differs. What is it, and why does that make the accuracy comparison fair?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the shared skeleton: message passing with message &rarr; aggregate &rarr; update. — _Holding the skeleton fixed isolates the variable of interest._
- Name the one moving part: the aggregation step &mdash; fixed degree-normalized average (GCN) vs sampled mean + concat (GraphSAGE) vs learned attention (GAT). — _Any accuracy gap must come from the aggregator, not from a luckier split or optimizer._

**Answer:** Only the neighbor-aggregation differs. Because everything else is held identical, a difference in test accuracy is attributable to the aggregation style alone &mdash; a controlled comparison.

</details>

**Problem 2.** In our run the node features are deliberately weak (nearly uninformative on their own), yet all three GNNs classify most nodes correctly. Where does the signal come from?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how the toy graph is built: edges connect same-class nodes far more often than different-class nodes (homophily). — _That structure puts the class signal in the edges, not the node features._
- Trace message passing: after two layers, each node's vector is a blend of its 2-hop neighborhood, which is dominated by its own class. — _Aggregation converts "who I am connected to" into a usable feature._

**Answer:** From the graph structure. The few labels diffuse along same-class edges through message passing, so even with weak features the neighborhood reveals the class. Remove the edges (aggregate only self) and accuracy collapses toward chance.

</details>